# Notebook 3: Model Training & Evaluation Pipeline

## Objective
In this notebook, we will train and compare various machine learning models to predict customer churn. 
We will focus heavily on detecting **Class 1 (Attrited Customer)**.

**Workflow:**
1. **Base Models Evaluation:** Compare Logistic Regression, Random Forest, XGBoost, LightGBM, and CatBoost.
2. **Cross-Validation:** Validate the stability of our top-performing models using a strict SMOTE Pipeline to prevent data leakage.
3. **Stacking Ensemble:** Combine the power of the best models to push the accuracy and F1-score to the absolute limit.

In [24]:
!pip install xgboost lightgbm catboost imbalanced-learn -q

Imports & Data Loading

In [2]:
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Load the 6 variables from the saved pickle file (Adjust path if needed)
try:
    with open('processed_data.pkl', 'rb') as f:
        X_train_balanced, y_train_balanced, X_test_scaled, y_test, X_train_scaled, y_train = pickle.load(f)
except FileNotFoundError:
    with open('../data/processed/processed_data.pkl', 'rb') as f:
        X_train_balanced, y_train_balanced, X_test_scaled, y_test, X_train_scaled, y_train = pickle.load(f)

# fix target labels: Convert String to Numeric (1 for Churn, 0 for Stay)
y_train_balanced = y_train_balanced.map({'Existing Customer': 0, 'Attrited Customer': 1})
y_test = y_test.map({'Existing Customer': 0, 'Attrited Customer': 1})
y_train = y_train.map({'Existing Customer': 0, 'Attrited Customer': 1})

print("Data loaded and target labels encoded successfully (Attrited = 1)!")
print(f"- Training Data Shape (Balanced): {X_train_balanced.shape}")
print(f"- Test Data Shape: {X_test_scaled.shape}")

Data loaded and target labels encoded successfully (Attrited = 1)!
- Training Data Shape (Balanced): (13598, 21)
- Test Data Shape: (2026, 21)


Train Base Models

In [3]:
# Define the base models
base_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0) # Added CatBoost here
}

# List to store results
base_results = []

print("Starting Base Models Evaluation...\n")

# Loop through, train, and evaluate
for name, model in base_models.items():
    print(f"Training {name}...")
    
    # Train
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate and store results
    base_results.append({
        "Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Churn)": precision_score(y_test, y_pred, pos_label=1),
        "Recall (Churn)": recall_score(y_test, y_pred, pos_label=1),
        "F1-Score (Churn)": f1_score(y_test, y_pred, pos_label=1)
    })

print("\nBase models evaluation completed!")

Starting Base Models Evaluation...

Training Logistic Regression...
Training Random Forest...
Training XGBoost...
Training LightGBM...
Training CatBoost...

Base models evaluation completed!


## Model Comparison & Leaderboard
Convert results into a DataFrame and sort them by **F1-Score** to identify the best performing model for predicting customer churn.

In [ ]:
# Create DataFrame and sort by F1-Score
leaderboard_df = pd.DataFrame(base_results).sort_values(by="F1-Score (Churn)", ascending=False).reset_index(drop=True)

# Highlight the best scores
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]

print("BASE MODELS LEADERBOARD:")
display(leaderboard_df.style.apply(highlight_max, subset=['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-Score (Churn)']))

BASE MODELS LEADERBOARD:


,Model Name,Accuracy,Precision (Churn),Recall (Churn),F1-Score (Churn)
0,CatBoost,0.974827,0.912651,0.932308,0.922374
1,XGBoost,0.972359,0.916409,0.910769,0.913580
2,LightGBM,0.969398,0.894895,0.916923,0.905775
3,Random Forest,0.963968,0.868421,0.913846,0.890555
4,Logistic Regression,0.864265,0.549407,0.855385,0.669073


## Strict Cross-Validation (Preventing Data Leakage)
To ensure our top models are not overfitting, we will run a **5-Fold Stratified Cross-Validation**. 
*Crucial:* We use an `imbalanced-learn Pipeline` to apply SMOTE *during* each fold, not before, ensuring the validation fold remains completely unseen and synthetic-free.

In [5]:
# Select the top 3 boosting models for CV
cv_models = {
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1']

print("Running Strict 5-Fold Cross-Validation...\n")

for name, model in cv_models.items():
    print(f"Cross-Validating {name}...")
    
    # Create the Pipeline: SMOTE -> Model
    pipeline = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    
    # Pass the original scaled data (X_train_scaled, y_train), NOT the already balanced one
    cv_results = cross_validate(pipeline, X_train_scaled, y_train, cv=skf, scoring=scoring)
    
    print(f"{name} True Performance (5-Fold Average):")
    print(f"   - F1-Score : {np.mean(cv_results['test_f1']):.4f}")
    print(f"   - Recall   : {np.mean(cv_results['test_recall']):.4f}")
    print(f"   - Precision: {np.mean(cv_results['test_precision']):.4f}\n")

Running Strict 5-Fold Cross-Validation...

Cross-Validating LightGBM...
LightGBM True Performance (5-Fold Average):
   - F1-Score : 0.9068
   - Recall   : 0.9071
   - Precision: 0.9071

Cross-Validating XGBoost...
XGBoost True Performance (5-Fold Average):
   - F1-Score : 0.9049
   - Recall   : 0.8979
   - Precision: 0.9123

Cross-Validating CatBoost...
CatBoost True Performance (5-Fold Average):
   - F1-Score : 0.9107
   - Recall   : 0.9040
   - Precision: 0.9176



## Stacking Ensemble
Finally, we will combine our top-performing tree-based models. A Meta-Model (Logistic Regression) will learn from the predictions of LightGBM, XGBoost, and CatBoost to make the final, highly accurate prediction.

In [6]:
print("Initializing Stacking Ensemble...\n")

# Define the base learners (Level-0 models)
level0_estimators = [
    ('lgbm', LGBMClassifier(random_state=42, verbose=-1)),
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss')),
    ('cat', CatBoostClassifier(random_state=42, verbose=0))
]

# Define the meta-model (Level-1 model)
level1_estimator = LogisticRegression()

# Create the Stacking model
stacking_model = StackingClassifier(estimators=level0_estimators, final_estimator=level1_estimator)

# Train and Evaluate
print("Training Stacking Ensemble (This might take a moment, just chill...)")
stacking_model.fit(X_train_balanced, y_train_balanced)

y_pred_stack = stacking_model.predict(X_test_scaled)

# Final results
print()
print("="*50)
print("FINAL STACKING ENSEMBLE RESULTS (TEST SET)")
print("="*50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_stack):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_stack, pos_label=1):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_stack, pos_label=1):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_stack, pos_label=1):.4f}")
print("="*50)

Initializing Stacking Ensemble...

Training Stacking Ensemble (This might take a moment, just chill...)

FINAL STACKING ENSEMBLE RESULTS (TEST SET)
Accuracy : 0.9738
Precision: 0.9024
Recall   : 0.9385
F1-Score : 0.9201


## Conclusion & Key Takeaways
We have successfully built a robust machine learning pipeline to predict customer churn. 
By utilizing an `ImbPipeline` with **SMOTE**, we completely eliminated data leakage during Cross-Validation. The results prove that our top tree-based models (CatBoost, LightGBM, XGBoost) and the **Stacking Ensemble** are highly stable, generalizing well to unseen data without overfitting. We have achieved a highly reliable F1-Score, balancing both Precision (cost-efficiency) and Recall (retention effectiveness).

---

## Next Steps (Plan for Next Week)
To push this project from a "good model" to an "industry-grade solution," we will focus on optimization and model interpretability in the upcoming week:

### 1. Advanced Hyperparameter Tuning
While our models perform well with default settings, they are not yet operating at their absolute limits.
* **Action:** We will use **Optuna** or **GridSearchCV** to systematically search for the optimal hyperparameter combinations (e.g., `learning_rate`, `max_depth`, `num_leaves`) for our champion model (CatBoost or LightGBM).
* **Goal:** Maximize the **F1-Score** and push the model's predictive power to its ceiling.

### 2. Decision Threshold Tuning
By default, the model classifies a customer as a churner if the probability is $> 0.5$. However, from a business perspective, the cost of losing a customer is often higher than the cost of a retention campaign.
* **Action:** We will analyze the **Precision-Recall Curve** to find an optimal custom threshold (e.g., $> 0.35$).
* **Goal:** Increase **Recall** to capture even more potential churners without sacrificing too much Precision.

### 3. Model Interpretability (Feature Importance & SHAP)
Predicting *who* will churn is only half the battle; the bank needs to know *why* they are leaving to take actionable measures.
* **Action:** We will calculate the inherent **Feature Importance** of our model and utilize **SHAP (SHapley Additive exPlanations)** values  for deep, local interpretability.
* **Goal:** Discover the key drivers of customer attrition (e.g., dropping transaction counts, specific demographic combinations) to help the Marketing team design highly targeted retention strategies.